# 03 - Validation (Phases 7 and 9)
Foreign keys, join safety and post-cleaning quality checks, run on the cleaned CSVs re-read from disk. Logic lives in `src/validation.py`; results are appended to `reports/data_quality_report.xlsx`.

In [1]:
import sys; sys.path.insert(0, '..')
import pandas as pd
from IPython.display import Image, display
pd.set_option('display.max_columns', 40, 'display.max_colwidth', 90, 'display.max_rows', 80)
from src.validation import run_validation
res = run_validation()

## Foreign-key validation (cleaned data)

In [2]:
res['fk_validation_clean']

,relationship,expected_cardinality,child_rows,parent_rows,missing_fk,orphan_child_rows,orphan_distinct_keys,parents_without_children,max_children_per_parent,parent_key_unique,verdict
0,orders.customer_id -> customers.customer_id,1:1,99441,99441,0,0,0,0,1,True,PASS
1,order_items.order_id -> orders.order_id,N:1,112650,99441,0,0,0,775,21,True,PASS
2,order_items.product_id -> products.product_id,N:1,112650,32951,0,0,0,0,527,True,PASS
3,order_items.seller_id -> sellers.seller_id,N:1,112650,3095,0,0,0,0,2033,True,PASS
4,payments.order_id -> orders.order_id,N:1,103886,99441,0,0,0,1,29,True,PASS
5,reviews.order_id -> orders.order_id,N:1,99224,99441,0,0,0,768,3,True,PASS
6,products.product_category_name -> category_translation.product_category_name,N:1,32951,73,0,0,0,0,3029,True,PASS
7,customers.customer_zip_code_prefix -> geo_zip.zip_code_prefix,N:1,99441,19015,0,278,157,4178,142,True,ACCEPTED GAP (optional reference)
8,sellers.seller_zip_code_prefix -> geo_zip.zip_code_prefix,N:1,3095,19015,0,7,7,16776,49,True,ACCEPTED GAP (optional reference)


## Why tables must be aggregated before joining
Items and payments are both many-per-order. Joining them directly multiplies rows and inflates revenue; aggregating each to order grain first reproduces the true totals exactly.

In [3]:
res['join_risk']

,measure,value,note
0,orders with >1 item row,9803.00,
1,orders with >1 payment row,2961.00,
2,orders with >1 item AND >1 payment (many-to-many on order_id),275.00,unexpected M:N if joined directly
3,orders with >1 review row,547.00,
4,"TRUE item value (sum of price, item grain)",13591643.70,reference
5,"TRUE payment total (sum of payment_value, payment grain)",16008872.12,reference
6,NAIVE items x payments join - rows,117601.00,vs 112650 item rows
7,NAIVE join - sum(price),14209115.34,inflated by 4.54%
8,NAIVE join - sum(payment_value),20308134.71,inflated by 26.86%
9,NAIVE items x payments x reviews - sum(payment_value),20187928.70,inflated by 26.10%


## Post-cleaning checks

In [4]:
v = res['post_clean_validation']
print(v.result.value_counts().to_string())
v.groupby(['area', 'result']).size().unstack(fill_value=0)

result
PASS    95


result,PASS
area,
categoricals,18
data types,11
dates,6
duplicates,10
foreign key,9
missing values,11
outliers,1
primary key,9
ranges,6


In [5]:
v[v.result == 'FAIL']

,area,dataset,check,result,detail


## Raw vs cleaned

In [6]:
res['raw_vs_clean']

,dataset,raw_rows,clean_rows,rows_removed,rows_added,raw_columns,clean_columns,columns_added,raw_missing_cells,clean_missing_cells,dtype_changes
0,orders,99441,99441,0,0,8,11,"flag_status_date_conflict, flag_date_sequence_anomaly, flag_carrier_before_approval",4908,4908,8
1,order_items,112650,112650,0,0,7,10,"flag_shipping_limit_anomaly, price_outlier_iqr, freight_outlier_iqr",0,0,4
2,payments,103886,103886,0,0,5,6,payment_value_outlier_iqr,0,0,2
3,reviews,99224,99224,0,0,7,11,"has_comment_title, has_comment_message, flag_review_before_purchase, is_latest_review_...",145903,145932,6
4,customers,99441,99441,0,0,5,5,,0,0,5
5,products,32951,32951,0,0,9,13,"product_category_name_english, flag_category_missing, flag_physical_attrs_imputed, wei...",2448,1830,9
6,sellers,3095,3095,0,0,4,4,,0,0,4
7,geolocation,1000163,720492,279671,0,5,7,"flag_outside_brazil, dist_to_prefix_median_km",0,0,3
8,category_translation,71,73,0,2,2,2,,0,0,2
9,geo_zip (new),0,19015,0,19015,0,11,(postal-prefix reference table),0,16,0


In [7]:
res['cleaning_summary']

,metric,value
0,Original row count (9 raw files),1550922
1,Final row count (9 cleaned files),1271253
2,Rows removed - geolocation duplicates only,279671
3,"Transactional rows removed (orders, items, payments, reviews, customers, products, sel...",0
4,Duplicate records removed,279671
5,"Missing values treated (labelled or imputed, each with an indicator flag)",618
6,Missing values deliberately preserved,152670
7,"Invalid values corrected (0 g weights 4, 0 instalments 2, seller cities, geolocation s...",75
8,"Invalid records excluded from an aggregate (out-of-Brazil points, still kept in the ro...",27
9,Outliers flagged (IQR) and retained,33093


## Remaining data-quality issues (documented, not hidden)

In [8]:
res['remaining_issues']

,dataset,issue,flag_available,recommended_handling
0,orders,8 delivered orders have no delivery date; 6 canceled orders have one,flag_status_date_conflict,exclude via flag in delivery-time analysis
1,orders,189 orders with impossible chronology (carrier before purchase / delivery before carrier),flag_date_sequence_anomaly,exclude via flag when computing stage durations
2,orders,Edge months are incomplete (2016-09..2016-12 and 2018-09..2018-10 hold 25+ orders only...,none,restrict trend/forecast work to 2017-01 .. 2018-08
3,orders,775 orders have no items; 1 delivered order has no payment record; 768 orders have no ...,has_items / missing values in sales table,inherent to the extract; not recoverable
4,payments,"Payment total differs from items + freight for ~260 orders (instalment interest, vouch...",payment_mismatch in sales table,choose one revenue basis and state it; do not mix
5,payments,80 orders have gaps in payment_sequential (records absent from the extract),none,payment totals for those orders may be understated
6,products,610 products have no category ('unknown') and no listing metadata,flag_category_missing,report 'unknown' as its own group
7,reviews,"A review belongs to an order, not to a product or seller",none,product/seller satisfaction is an approximation for multi-item orders
8,reviews,64 reviews predate the purchase of the order they are attached to,flag_review_before_purchase,exclude when modelling review timing
9,geolocation,"279 customers and 7 sellers have no coordinate; 2,094 prefixes rest on <3 points or wi...",flag_low_confidence,distance features are missing/less reliable for these
